# Import libraries

In [2]:
import pandas as pd
import itertools
from itertools import combinations
from tqdm import tqdm 
import csv

# Initial checks

## Column values verification -> 'location' is a concatenation of 'latitude' and 'longitude'

In [4]:
df = pd.read_csv(
    "Food_Inspections_20260328.csv",
)

df = df.dropna(subset=["Latitude", "Longitude", "Location"])

# Reconstruct what Location should look like if it's just lat/long combined
df["expected_location"] = "(" + df["Latitude"].astype(str) + ", " + df["Longitude"].astype(str) + ")"

# Compare against actual
mismatches = df[df["expected_location"] != df["Location"]]

print(f"Total rows checked : {len(df):,}")
print(f"Mismatches         : {len(mismatches):,}")

if len(mismatches) > 0:
    print("\nSample mismatches:")
    print(mismatches[["Latitude", "Longitude", "Location", "expected_location"]].head())
else:
    print("\nLocation is exactly a concatenation of Latitude and Longitude")

Total rows checked : 306,836
Mismatches         : 306,836

Sample mismatches:
    Latitude  Longitude                                 Location  \
0  41.950348 -87.807329  (41.95034769269968, -87.80732944580947)   
1  41.953258 -87.769512  (41.95325816953041, -87.76951202741661)   
2  41.910341 -87.679283   (41.9103413262732, -87.67928260452388)   
3  41.868784 -87.686245  (41.86878383901946, -87.68624468790969)   
4  41.943611 -87.654369   (41.9436113646505, -87.65436890530064)   

                   expected_location  
0   (41.9503476927, -87.80732944581)  
1  (41.95325816953, -87.76951202742)  
2  (41.91034132627, -87.67928260452)  
3  (41.86878383902, -87.68624468791)  
4   (41.94361136465, -87.6543689053)  


## Check unique values across columns, especially in 'city' -> mostly are Chicago varied with typos

In [5]:
# Print unique values for the City column
# clean whitespace and casing to get a true count
unique_cities = df['City'].dropna().str.upper().str.strip().unique()

print("Unique Values in [City]")
print(unique_cities)
print(f"\nTotal unique cities found: {len(unique_cities)}")

# Print unique value counts per column
cols = df.columns
unique_counts = {col: df[col].nunique() for col in cols}

print("\nUnique value counts per column:")
# Sorted by highest number of unique values first
for col, n in sorted(unique_counts.items(), key=lambda x: -x[1]):
    print(f"  {col:40s}: {n:,}")

Unique Values in [City]
['CHICAGO' 'CCHICAGO' 'CHICAGOO' 'CH' '312CHICAGO' 'CHICAGOCHICAGO'
 'CHICAGO.' 'BERWYN' 'CHICAGOC' 'CHICAGOBEDFORD PARK' 'CHCICAGO'
 'CHARLES A HAYES' 'CHCHICAGO' 'CHICAGOI' 'SUMMIT' 'WESTMONT' 'LOMBARD'
 'INACTIVE' 'ALSIP' 'BLUE ISLAND']

Total unique cities found: 20

Unique value counts per column:
  Inspection ID                           : 306,836
  Violations                              : 219,199
  License #                               : 48,036
  DBA Name                                : 34,432
  Address                                 : 32,823
  AKA Name                                : 32,769
  Latitude                                : 18,810
  Longitude                               : 18,810
  Location                                : 18,810
  expected_location                       : 18,810
  Inspection Date                         : 4,085
  Facility Type                           : 515
  Inspection Type                         : 110
  Zip         

# Main

## Read data

In [8]:
# 1. Read data
df = pd.read_csv(
    "Food_Inspections_20260328.csv",
)

# Drop free-text and derived/duplicate columns
DROP_COLS = ["Violations", "Location"]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

# Drop rows with nulls (nulls create false FD violations)
df = df.dropna()

print(f"  Full dataset : {len(df):,} rows, {len(df.columns)} columns")
print(f"  Columns      : {list(df.columns)}\n")

cols = list(df.columns)

  Full dataset : 299,095 rows, 15 columns
  Columns      : ['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type', 'Risk', 'Address', 'City', 'State', 'Zip', 'Inspection Date', 'Inspection Type', 'Results', 'Latitude', 'Longitude']



## Find FDs

In [9]:
# 2. Sample for discovery, keep full df for verification
# SAMPLE_SIZE = 150_000
# df_sample = df.sample(min(SAMPLE_SIZE, len(df)), random_state=42)
# print(f"  Sample size  : {len(df_sample):,} rows\n")

df_sample = df

# 3. Precompute unique counts per column (used for pruning)
unique_counts = {col: df_sample[col].nunique() for col in cols}
print("Unique value counts per column:")
for col, n in sorted(unique_counts.items(), key=lambda x: -x[1]):
    print(f"  {col:40s}: {n:,}")
print()

# 4. Core FD check
def holds(df, lhs: list, rhs: str) -> bool:
    # Returns True if LHS → RHS holds (each LHS group maps to exactly 1 RHS value)
    return (df.groupby(lhs)[rhs].nunique() <= 1).all()

# 5. Composite unique count for pruning multi-col LHS
def lhs_unique(df, lhs: list) -> int:
    # Number of unique combinations in a composite LHS
    return df.groupby(lhs).ngroups

# 6. Find all minimal non-trivial FDs
def find_minimal_nontrivial_fds(df, cols, max_lhs_size=3):
    """
    Finds all minimal, non-trivial FDs using three pruning strategies:

    Pruning 1 — Trivial check    : skip if rhs is already in lhs
    Pruning 2 — Unique count     : skip if nunique(lhs) < nunique(rhs)
                                   (impossible for FD to hold)
    Pruning 3 — Minimality       : skip if any proper subset of lhs
                                   already determines rhs (not minimal)
    """
    minimal_fds = []

    # Stores (frozenset(lhs), rhs) for all FDs found so far
    # Used for minimality pruning in larger LHS sizes
    known_fds = set()

    # outer loop : LHS size from 1 to max
    for lhs_size in range(1, max_lhs_size + 1):
        print(f"Checking LHS size {lhs_size}...")
        total_checked = 0
        pruned_trivial = 0
        pruned_count   = 0
        pruned_minimal = 0
        found          = 0

        # inner loop 1: take LHS
        for lhs in combinations(cols, lhs_size):
            lhs_set    = frozenset(lhs)
            lhs_n_uniq = lhs_unique(df, list(lhs))

            # inner loop 2: take RHS
            for rhs in cols:

                # Pruning 1: Trivial (RHS already in LHS)
                if rhs in lhs_set:
                    pruned_trivial += 1
                    continue

                # Pruning 2: Unique count
                # If LHS has fewer unique combos than RHS unique values,
                # it's impossible for LHS → RHS to hold
                if lhs_n_uniq < unique_counts[rhs]:
                    pruned_count += 1
                    continue

                # Pruning 3: Minimality
                # If any proper subset of LHS already determines RHS,
                # this FD is not minimal - skip
                is_minimal = True
                for subset_size in range(1, lhs_size):
                    # take subset and check if already exists valid FDs
                    for subset in combinations(lhs, subset_size):
                        # if subset is already in known FDs = not minimal -> skip
                        if (frozenset(subset), rhs) in known_fds:
                            is_minimal = False
                            break
                    if not is_minimal:
                        break

                if not is_minimal:
                    pruned_minimal += 1
                    continue

                # Expensive check only if all pruning passed
                total_checked += 1
                if holds(df, list(lhs), rhs):
                    minimal_fds.append({
                        "lhs":      list(lhs),
                        "rhs":      rhs,
                        "lhs_size": lhs_size
                    })
                    known_fds.add((lhs_set, rhs))
                    found += 1

        print(f"  Pruned - trivial: {pruned_trivial:,} | "
              f"Pruned - unique count: {pruned_count:,} | "
              f"Pruned- minimality: {pruned_minimal:,}")
        print(f"  FD candidates passed pruning and checked with sample dataset: {total_checked:,} | Valid FDs Found: {found}\n")

    return pd.DataFrame(minimal_fds)

fds_sample = find_minimal_nontrivial_fds(df_sample, cols, max_lhs_size=3)

verified_df = fds_sample

# 7. Verify on full dataset
# print(f"Verifying {len(fds_sample)} FDs on full dataset ({len(df):,} rows)...")

# verified_rows = []
# for _, row in fds_sample.iterrows():
#     # check if FD also holds in full dataset
#     full_holds = holds(df, row["lhs"], row["rhs"])
#     # append if verified
#     verified_rows.append({**row, "holds_on_full": full_holds})

# verified_df = pd.DataFrame(verified_rows)

# # Print figures
# sample_only = (~verified_df["holds_on_full"]).sum() # FDs that hold only on sample
# confirmed   = verified_df["holds_on_full"].sum() # FDs that hold on full dataset

# print(f"  Confirmed on full data : {confirmed}")
# print(f"  Sample only (rejected) : {sample_only}\n")

# 8. Final results
strict = verified_df.copy()

print("=" * 60)
print(f"ALL MINIMAL NON-TRIVIAL FDs ({len(strict)} total)")
print("=" * 60)

for size in range(1, 4):
    group = strict[strict["lhs_size"] == size]
    if len(group) == 0:
        continue
    print(f"\n── LHS size {size} ({len(group)} FDs) ──")
    for _, row in group.iterrows():
        print(f"  {row['lhs']}  →  '{row['rhs']}'")

# 9. Save
strict[["lhs", "rhs", "lhs_size"]].to_csv("minimal_FDs.csv", index=False)
print("\nSaved to: minimal_FDs.csv")

Unique value counts per column:
  Inspection ID                           : 299,095
  License #                               : 42,572
  Address                                 : 31,335
  DBA Name                                : 29,960
  AKA Name                                : 29,045
  Latitude                                : 17,713
  Longitude                               : 17,713
  Inspection Date                         : 4,084
  Facility Type                           : 490
  Inspection Type                         : 101
  Zip                                     : 65
  City                                    : 20
  Results                                 : 7
  Risk                                    : 4
  State                                   : 1

Checking LHS size 1...
  Pruned - trivial: 15 | Pruned - unique count: 104 | Pruned- minimality: 0
  FD candidates passed pruning and checked with sample dataset: 106 | Valid FDs Found: 31

Checking LHS size 2...
  Pruned - trivial